In [3]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("../datasets/ai4i2020.csv")

In [5]:
X = df.drop(
    columns=[
        "UDI",
        "Product ID",
        "Machine failure",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF",
    ]
)

y = df["Machine failure"]

In [6]:
categorical_features = ["Type"]

numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_features,
        ),
    ]
)

In [7]:
negative = (y == 0).sum()
positive = (y == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

28.49852507374631


In [8]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=500,
                max_depth=6,
               learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss",
            ),
        ),
    ]
)

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [10]:
scores = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
    ],
)

In [11]:
print("Average Accuracy :", scores["test_accuracy"].mean())
print("Average Precision:", scores["test_precision"].mean())
print("Average Recall   :", scores["test_recall"].mean())
print("Average F1 Score :", scores["test_f1"].mean())

Average Accuracy : 0.9808
Average Precision: 0.7094949389812778
Average Recall   : 0.7376207199297629
Average F1 Score : 0.7214645451438444


In [12]:
print("Accuracy Std :", scores["test_accuracy"].std())
print("Precision Std:", scores["test_precision"].std())
print("Recall Std   :", scores["test_recall"].std())
print("F1 Std       :", scores["test_f1"].std())

Accuracy Std : 0.0012884098726725086
Precision Std: 0.020622491665444706
Recall Std   : 0.06282247951830397
F1 Std       : 0.02773606733892845
